In [1]:
import pandas as pd
import numpy as np

## Load Dataset

In [4]:
df = pd.read_csv("HHS_Unaccompanied_Alien_Children_Program.csv")

## Convert Date & Sort by date

In [5]:
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

In [6]:
df.head()

,Date,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care
0,2023-01-12,33.0,53.0,34.0,"6,566",436.0
1,2023-01-22,32.0,49.0,39.0,"7,122",227.0
2,2023-01-23,32.0,50.0,39.0,"7,280",181.0
3,2023-01-24,47.0,42.0,47.0,"7,433",175.0
4,2023-01-25,20.0,22.0,41.0,"7,538",180.0


## Rename Columns


In [12]:
df.rename(columns={
    "Children apprehended and placed in CBP custody*": "CBP_Intake",
    "Children in CBP custody": "CBP_Custody",
    "Children transferred out of CBP custody": "Transferred_to_HHS",
    "Children in HHS Care": "HHS_Care",
    "Children discharged from HHS Care": "Discharged"
}, inplace=True)

## Pipeline Stages


In [8]:
df["Stage_1_CBP"] = df["CBP_Custody"]
df["Stage_2_HHS"] = df["HHS_Care"]
df["Stage_3_Sponsor"] = df["Discharged"]

## Daily Pipeline Movements


In [9]:

df["Movement_CBP_to_HHS"] = df["Transferred_to_HHS"]
df["Movement_HHS_to_Sponsor"] = df["Discharged"]


In [19]:
print(df.dtypes)

Date                       datetime64[us]
CBP_Intake                        float64
CBP_Custody                       float64
Transferred_to_HHS                float64
HHS_Care                          float64
Discharged                        float64
Stage_1_CBP                       float64
Stage_2_HHS                           str
Stage_3_Sponsor                   float64
Movement_CBP_to_HHS               float64
Movement_HHS_to_Sponsor           float64
Transfer_Efficiency_%             float64
Discharge_Efficiency_%            float64
Pipeline_Throughput_%             float64
CBP_Backlog_Change                float64
HHS_Backlog_Change                float64
Pipeline_Pressure                 float64
Placement_Ratio_%                 float64
dtype: object


## Converting all columns to numeric

In [17]:
numeric_cols = [
    "CBP_Intake",
    "CBP_Custody",
    "Transferred_to_HHS",
    "HHS_Care",
    "Discharged"
]

for col in numeric_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.strip()
    )

    df[col] = pd.to_numeric(df[col], errors="coerce")

## KPI Calculations
- Transfer Efficiency
- Discharge Efficiency
- Pipeline Throughout
- CBP Backlog Change
- HHS Backlog Change
- Pipeline Pressure
- Placement Ratio

In [18]:
df["Transfer_Efficiency_%"] = (
    df["Transferred_to_HHS"] /
    df["CBP_Intake"]
) * 100

df["Discharge_Efficiency_%"] = (
    df["Discharged"] /
    df["HHS_Care"]
) * 100


df["Pipeline_Throughput_%"] = (
    df["Discharged"] /
    df["CBP_Intake"]
) * 100


df["CBP_Backlog_Change"] = (
    df["CBP_Intake"] -
    df["Transferred_to_HHS"]
)


df["HHS_Backlog_Change"] = (
    df["Transferred_to_HHS"] -
    df["Discharged"]
)


df["Pipeline_Pressure"] = (
    df["CBP_Custody"] +
    df["HHS_Care"]
)


df["Placement_Ratio_%"] = (
    df["Discharged"] /
    df["Transferred_to_HHS"]
) * 100

## Rolling Averages


In [20]:
df["Intake_7Day_Avg"] = (
    df["CBP_Intake"]
    .rolling(7, min_periods=1)
    .mean()
)

df["Transfer_7Day_Avg"] = (
    df["Transferred_to_HHS"]
    .rolling(7, min_periods=1)
    .mean()
)

df["Discharge_7Day_Avg"] = (
    df["Discharged"]
    .rolling(7, min_periods=1)
    .mean()
)

## Monthly Summary


In [21]:
df["Month"] = df["Date"].dt.to_period("M")

monthly_summary = (
    df.groupby("Month")[
        [
            "CBP_Intake",
            "Transferred_to_HHS",
            "HHS_Care",
            "Discharged",
            "Pipeline_Pressure"
        ]
    ]
    .sum()
)

## Pipeline Overview

In [22]:
summary = {
    "Total Intake": df["CBP_Intake"].sum(),
    "Total CBP Transfers": df["Transferred_to_HHS"].sum(),
    "Total HHS Population": df["HHS_Care"].sum(),
    "Total Discharged": df["Discharged"].sum(),
    "Average Transfer Efficiency (%)": round(df["Transfer_Efficiency_%"].mean(), 2),
    "Average Discharge Efficiency (%)": round(df["Discharge_Efficiency_%"].mean(), 2),
    "Average Pipeline Throughput (%)": round(df["Pipeline_Throughput_%"].mean(), 2),
    "Maximum Pipeline Pressure": df["Pipeline_Pressure"].max()
}

print("\n========== PIPELINE SUMMARY ==========\n")
for k, v in summary.items():
    print(f"{k}: {v}")

print("\n========== MONTHLY SUMMARY ==========\n")
print(monthly_summary)


========== PIPELINE SUMMARY ==========

Total Intake: 67337.0
Total CBP Transfers: 92641.0
Total HHS Population: 4364118.0
Total Discharged: 124853.0
Average Transfer Efficiency (%): inf
Average Discharge Efficiency (%): 2.37
Average Pipeline Throughput (%): inf
Maximum Pipeline Pressure: 11762.0

========== MONTHLY SUMMARY ==========

         CBP_Intake  Transferred_to_HHS  HHS_Care  Discharged  \
Month                                                           
2023-01       247.0               276.0   58957.0      1856.0   
2023-02      1886.0              2637.0  148120.0      5183.0   
2023-03      2479.0              3399.0  150962.0      5375.0   
2023-04      3129.0              3967.0  164696.0      5668.0   
2023-05      2746.0              3622.0  180117.0      6256.0   
2023-06      1774.0              2416.0  122729.0      4029.0   
2023-07      2858.0              3897.0  143390.0      4407.0   
2023-08      2751.0              3587.0  186958.0      6470.0   
2023-09    

In [ ]:
# Saving Results

df.to_csv("Care_Transition_Pipeline_Analysis.csv", index=False)

print("\nAnalysis completed successfully!")
print("Output saved as 'Care_Transition_Pipeline_Analysis.csv'")


Analysis completed successfully!
Output saved as 'Care_Transition_Pipeline_Analysis.csv'
